In [3]:
!pip install evaluate
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.6 MB/s eta 0:00:00


In [4]:
# Imports
from dataclasses import dataclass
import re
import string
from typing import Any, Dict, List, Union

import evaluate
import numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq


# Fine-tuning T5 for German complex word identification

## 1. Loading the model

Here, I work with T5, a translation model supporting multiple languages off-the-shelf. Good for sentence2sentence mapping.

In [16]:
model_name = "google-t5/t5-small"

# Load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [17]:
# Set up data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_name)

## 2. Loading the dataset

I'll be using my self-made dataset from German MDR news articles.

In [5]:
# Get dataset
DATAFILE = "mdr_4_unified.json"

dataset = load_dataset("json", data_files=DATAFILE, field="data")

# Split into train, dev and test
ds = dataset["train"].train_test_split(test_size=0.2)
print(ds)


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'target'],
        num_rows: 1345
    })
    test: Dataset({
        features: ['sentence', 'target'],
        num_rows: 337
    })
})


In [6]:
# Inspect items
for i in ds["train"]:
    print("sentence:", i["sentence"])
    print("target:", i["target"])
    break

sentence: Wer linke Gruppen, Antifaschisten und Klimaaktivisten mit Rechtsextremen gleichsetze, verkenne den Ernst der Lage.
target: Wer [linke] Gruppen, [Antifaschisten] und Klimaaktivisten mit [Rechtsextremen] [gleichsetze], [verkenne] den [Ernst] der [Lage].


The dataset is then tokenized to prepare for fine-tuning. I'll also add a prefix for the task at hand. (Just because.)

In [10]:
# prepare the model input
prefix = "Identify complex words:"

def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["sentence"]]
    model_inputs = tokenizer(inputs)

    labels = tokenizer(text_target=examples["target"])

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

ds_tokenized = ds.map(preprocess_function, batched=True)
ds_tokenized = ds_tokenized.filter(lambda x: len(x["input_ids"]) < 1025)

print(ds_tokenized)


Map:   0%|          | 0/1345 [00:00<?, ? examples/s]

Map:   0%|          | 0/337 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1345 [00:00<?, ? examples/s]

Filter:   0%|          | 0/337 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1345
    })
    test: Dataset({
        features: ['sentence', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 337
    })
})


## 3. Create evaluation metrics

Generally, I feel like precision and recall make a lot of sense for complex word identification. They do, however, need a couple of work arounds to not
throw division-by-zero errors in this setup. Also, T5 can and will change the
output sentence, which is not exactly a desired outcome. The word error rate accounts for this. I have decided to also calculate a mean score from these three metrics to provide a unified picture of model performance.


In [11]:
# Load metrics
wer = evaluate.load("wer")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

# Pattern for retrieving targets
pattern = r"\[([^ ]+)\]"


def retrieve_targets(items):
    """Find all targets in a given sequence."""
    return [re.findall(pattern, s) for s in items]

def postprocess_for_prec_rec(preds, labels):
    # Well one could do this elegantly with re, but this also works
    preds = [[1 if "]" in word else 0 for word in p.split()] for p in preds]
    labels = [[1 if "]" in word else 0 for word in l.split()] for l in labels]
    return preds, labels


def treat_double_words(decoded_str_list):
    """Treat double occurrences of words by numerating them."""
    returns = []
    for decoded_str in decoded_str_list:
        set_words = {}
        # Most punctuation will not be needed henceforth
        decoded_str = decoded_str.translate(str.maketrans("", "", "!\"#$%&'()*+,-./:;<=>?@\\^_`{|}~"))

        decoded_str = decoded_str.split()
        for i in range(len(decoded_str)):
            # Word counter
            if decoded_str[i] in set_words:
                set_words[decoded_str[i]] += 1
            else:
                set_words[decoded_str[i]] = 0
            # Treat marked words
            if decoded_str[i].endswith("]"):
                decoded_str[i] = decoded_str[i][:-1] + f"_{set_words[decoded_str[i]]}]"
            else:
                decoded_str[i] = decoded_str[i] + f"_{set_words[decoded_str[i]]}"
        decoded_str = " ".join(decoded_str)
        returns.append(decoded_str)
    return returns


def compute_metrics(eval_preds):
    """Compute metrics for evaluation loop.

    Main component is a score combining WER (to check if the model changes the
    phrase, which it sadly does), precision and recall. For the calculation of
    the latter two, workarounds are needed for cases with no predictions or no
    correct targets.
    The individual scores and the generation length are returned as well

    Returns:
        dict: containing all metrics
    """
    preds, labels = eval_preds
    # Decode predictions
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Decode labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Treat double occurrences of words
    decoded_labels = treat_double_words(decoded_labels)
    decoded_preds = treat_double_words(decoded_preds)

    # Get targets
    predicted_targets = retrieve_targets(decoded_preds)
    correct_targets = retrieve_targets(decoded_labels)

    # Calculate precision and recall
    precisions, recalls = [], []
    for c_targets, p_targets in zip(correct_targets, predicted_targets):
        tp = [t for t in p_targets if t in c_targets]
        # Treat edge case: precision cannot be calculated
        if len(p_targets) == 0:
            if len(p_targets) == 0:
                precisions.append(1)
            else:
                precisions.append(0)
        else:
            precisions.append(len(tp) / len(p_targets))
        # Treat edge case: recall cannot be calculated
        if len(c_targets) == 0:
            if len(c_targets) == 0:
                recalls.append(1)
            else:
                recalls.append(0)
        else:
            recalls.append(len(tp) / len(c_targets))

    # Postprocessing for WER computation
    decoded_preds = [
        i.translate(str.maketrans("", "", "!\"#$%&'()*+,-./:;<=>?@\\[]^_`{|}~"))
        for i in decoded_preds
    ]
    decoded_labels = [
        i.translate(str.maketrans("", "", "!\"#$%&'()*+,-./:;<=>?@\\[]^_`{|}~"))
        for i in decoded_labels
    ]
    # Calculate individual WERs
    wers = [wer.compute(predictions=[d], references=[l]) for d, l in zip(decoded_preds, decoded_labels)]

    # Calculate the mean of all three scores for a unified view
    res = [(p+r+(1-w))/3 for p, r, w in zip(precisions, recalls, wers)]

    result = {
        "3score": np.mean(res),
        "wer": np.mean(wers),
        "precision": np.mean(precisions),
        "recall": np.mean(recalls)
    }

    # Collect prediction length as well
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result


## 4. Set up training arguments for the fine-tuning

I'll continue testing different options here...

In [18]:
training_args = Seq2SeqTrainingArguments(
    output_dir="german_cwidentifier",
    eval_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.1,
    save_total_limit=3,
    num_train_epochs=25,
    generation_max_length=1024,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized["train"],
    eval_dataset=ds_tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


## 5. Fine-tuning!

In [19]:
trainer.train()


Epoch,Training Loss,Validation Loss,3score,Wer,Precision,Recall,Gen Len
1,0.783417,0.334369,0.741700,0.066200,0.664800,0.626600,37.884300
2,0.466735,0.275661,0.782400,0.053300,0.740200,0.660100,37.207700
3,0.321343,0.238534,0.817500,0.052000,0.756000,0.748700,38.439200
4,0.271393,0.217558,0.830000,0.047600,0.774600,0.763000,38.278900
5,0.236779,0.209250,0.846300,0.037400,0.801800,0.774500,37.857600
6,0.187321,0.202957,0.850800,0.033300,0.813200,0.772300,37.991100
7,0.162324,0.199253,0.862500,0.033100,0.816900,0.803700,38.192900
8,0.153635,0.185863,0.871500,0.032800,0.814300,0.833000,38.744800
9,0.126782,0.192953,0.877200,0.028300,0.847800,0.812000,37.911000
10,0.110842,0.191250,0.880000,0.026600,0.853900,0.812600,37.988100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=16825, training_loss=0.1419030421060049, metrics={'train_runtime': 4823.1214, 'train_samples_per_second': 6.972, 'train_steps_per_second': 3.488, 'total_flos': 356717901348864.0, 'train_loss': 0.1419030421060049, 'epoch': 25.0})

## 6. Test example

Just one, to see how well this worked.

In [20]:
model = trainer.model

In [21]:
sentence = "Identify complex words: Aktuell arbeiten wegen der hohen Energiepreise die Brennöfen auch nicht mehr permanent, sondern in Etappen."
input_ids = tokenizer(sentence, return_tensors="pt").to(model.device)

output = model.generate(**input_ids, cache_implementation="static", max_new_tokens=1024)

print("-"*100)
print("Input:", sentence)
print("Model output:")
print(tokenizer.decode(output[0], skip_special_tokens=True))

W0530 13:19:48.220000 2320 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


----------------------------------------------------------------------------------------------------
Input: Identify complex words: Aktuell arbeiten wegen der hohen Energiepreise die Brennöfen auch nicht mehr permanent, sondern in Etappen.
Model output:
[Aktuell] arbeiten wegen der hohen [Energiepreise] die [Brenöfen] auch nicht mehr [permanent], sondern in [Etappen].
